# LangGraph · Kestrel Labs' Accounts-Payable Desk

Kestrel Labs makes lab instruments. Invoices arrive by email, and each one is paid, held or refused according to a written policy: match it against the purchase order and what actually arrived, get the right people to sign, never pay twice, never change bank details because an email asked.

What makes this a job for LangGraph is **waiting**. An invoice can sit for days — on a vendor sending a corrected invoice, on a budget owner opening their approvals, on someone phoning a vendor to check a bank change. A run that waits for people has to be something that can be put down and picked up again, by another process, after a restart. Chapter 2's `Runtime` cannot do that: a run lives in one process from start to finish.

The desk is built twice, from the same policy and the same prompts:

| | Built from |
|---|---|
| **LangGraph** | `ap_graph.py` — a graph with a checkpointer, `interrupt()` for people, an agent as a subgraph |
| **From scratch** | `ap_scratch.py` and `case.py` — a **case file** rather than a graph: chapter 6's shared board, every write kept in chapter 4's `Records`; chapter 1's `extract()`; chapter 6's form-filling agent on chapter 2's loop |

The inbox — seven emails, each written to take a different path through the policy:

| | The email | What the policy says |
|---|---|---|
| **A** | Brandt Optics, 3,440.00, second half of a split delivery | matches, under 5,000: pay automatically |
| **B** | Corvid Electronics, 10,300.00 | matches: the budget owner, Tomasz, signs |
| **C** | Brandt Optics at 90.30 a unit; the PO says 86.00 | investigate, ask the vendor for a corrected invoice, **wait**, check it again, Mira signs |
| **D** | Halvorsen Metals, 60 housings billed, 45 received | investigate, pay for the 45, Tomasz signs |
| **E** | a "payment reminder" for an invoice paid on 4 September | never pay twice: reply with the payment reference |
| **F** | "Corvid" asking for payment to a **new bank account**, from a lookalike domain | freeze; the clerk phones Corvid **on the number on file**; it is fraud |
| **G** | Pinecrest Office Supply, 684.00, no purchase order | the clerk finds who ordered it; they sign |

| File | What it is |
|---|---|
| `payables.py` | the company: vendors, POs, receipts, the inbox, the policy as code, the investigator's tools |
| `ap_graph.py` | the desk in LangGraph |
| `ap_scratch.py`, `case.py` | the desk from this repo's parts |
| `ap_stubs.py` | stand-ins for every model call |
| `ap_checks.py` | five production behaviours, each reproduced offline |
| `test_ap.py` | offline tests for all of it |

---
# Part 1 · Build

## The rule the whole desk follows

**A model reads, investigates and writes. Code decides.** Three things need a model: reading an email into fields, working out *why* an invoice does not match, and writing to a vendor. Everything else — the three-way match, the thresholds, duplicates, bank details, who signs — is `payables.check()` and `approvers()`, plain Python that both builds import:

```python
# payables.py
def approvers(invoice, amount=None, owner=None):
    """Who must sign, in order. Empty for a matched invoice under the automatic limit.

    A non-PO invoice has no PO to name its owner, so `owner` is whoever the clerk found, and they
    sign whatever the amount -- the automatic limit is for invoices that have been matched, and
    there is nothing to match these against.
    """
    amount = invoice["total"] if amount is None else amount
    po = PURCHASE_ORDERS.get(invoice.get("po_number", ""))
    if po is None:
        return [owner] if owner else []
    if amount <= AUTO_APPROVE_LIMIT:
        return []
    signers = [po["owner"]]
    if amount > CONTROLLER_LIMIT:
        signers.append("ines")
    return signers
```

`check()` returns findings like `('price', 'LA-50 billed at 90.30; PO-4493 says 86.00')`, and the *kind* of each finding is what the graph routes on.

## State

LangGraph's state is a `TypedDict`. Every node receives all of it and returns only the fields it changed. A field annotated with a **reducer** is combined with the update instead of replaced — `trail: Annotated[list, operator.add]` means a node returning `{"trail": ["checked"]}` adds a line.

```python
# ap_graph.py
class Case(TypedDict, total=False):
    key: str                      # which inbox email this thread is about
    email: str                    # the email being read -- replaced by the vendor's reply
    invoice: dict
    findings: list                # [{"kind": ..., "detail": ...}] from payables.check
    recommendation: dict
    amount: float                 # what will be paid, which is not always what was billed
    qty: float
    owner: str                    # a non-PO invoice's signer, as the clerk found them
    signers: list
    approvals: Annotated[list, operator.add]
    vendor_rounds: int
    outbox: Annotated[list, operator.add]
    trail: Annotated[list, operator.add]
    status: str
```

## Nodes that wait for people

`interrupt()` stops the run, saves it, and hands a question to whoever is driving the graph. When an answer comes back through `Command(resume=...)`, the node runs again **from its first line**, and this time `interrupt()` returns the answer. One signer per pass; the router loops back here until everyone has signed.

```python
# ap_graph.py
def request_approval(state):
    signed = [a["by"] for a in state.get("approvals", [])]
    signer = next(s for s in state["signers"] if s not in signed)
    answer = interrupt({"ask": "approval", "approver": signer, "to": signer, "packet": _packet(state)})
    return {"approvals": [{"by": signer, "approved": bool(answer.get("approved")),
                           "note": answer.get("note", "")}],
            "trail": [f"{signer} {'approved' if answer.get('approved') else 'refused'}"]}

def verify_bank(state):
    vendor = P.VENDORS[P.find_vendor(state["invoice"])]
    answer = interrupt({"ask": "callback", "to": "dev",
                        "question": f"Call {vendor['name']} on {vendor['phone']} (the number ON FILE, "
                                    f"not one from the email) and confirm this change.",
                        "findings": [f["detail"] for f in state["findings"]]})
    if not answer.get("verified"):
        # routing from inside a node, with Command, because a person just decided it
        return Command(goto=END, update={"status": "fraud", "amount": 0.0,
                                         "trail": [f"bank change refused: {answer.get('note', '')}"]})
    return Command(goto="hold", update={"trail": ["bank change confirmed by phone; vendor master "
                                                  "must be updated before this is paid"]})
```

`verify_bank` routes itself with `Command(goto=...)`, because the decision belongs to the person who just answered. The phone number in the question is the one **on file** — never the one in the email, which in invoice F belongs to the fraudster.

The vendor loop is two nodes, not one, and the reason is the re-execution above: if sending and waiting shared a node, resuming the wait would send the email again.

```python
# ap_graph.py
def query_vendor(state):
    invoice, rec = state["invoice"], state["recommendation"]
    body = draft(vendor=invoice["vendor_name"], invoice_number=invoice["invoice_number"],
                 po=invoice["po_number"], reason=rec["reason"])
    return {"outbox": [{"to": invoice["sender_email"], "body": body}],
            "vendor_rounds": state.get("vendor_rounds", 0) + 1,
            "trail": [f"wrote to {invoice['sender_email']}"]}

def await_vendor(state):
    # Its own node, separate from the one that sends. A node that is resumed runs again from
    # its first line, so a send placed before this interrupt would go out twice.
    reply = interrupt({"ask": "vendor", "to": state["invoice"]["sender_email"],
                       "question": "Waiting for a corrected invoice."})
    return {"email": reply, "trail": ["vendor replied"]}
```

## Routing is code

Every conditional edge is a plain function over the state. None of them asks a model anything.

```python
# ap_graph.py
def after_check(state):
    kinds = {f["kind"] for f in state["findings"]}
    if not kinds:
        return "request_approval" if state["signers"] else "pay"
    for kind, node in [("duplicate", "answer_duplicate"), ("bank", "verify_bank"),
                       ("no_po", "ask_clerk"), ("unknown_vendor", "hold"), ("wrong_po", "hold")]:
        if kind in kinds:
            return node
    return "investigate"

def after_investigation(state):
    action = state["recommendation"]["action"]
    if action == "query_vendor":
        return "query_vendor" if state.get("vendor_rounds", 0) < P.MAX_VENDOR_ROUNDS else "hold"
    if action == "approve_partial":
        return "request_approval" if state["signers"] else "pay"
    return "hold"

def after_approval(state):
    if not state["approvals"][-1]["approved"]:
        return "refused"
    signed = {a["by"] for a in state["approvals"] if a["approved"]}
    return "pay" if signed >= set(state["signers"]) else "request_approval"
```

## The investigator: an agent as a subgraph

The investigator is a small graph of its own: `agent → tools → agent ...`, ending when the model calls `recommend` — a tool whose arguments **are** the answer (`payables.Recommendation`), so nothing ever executes it; the router sees its name and leaves the loop. It is built by hand rather than with LangGraph's `create_react_agent`, which 1.x deprecates in favour of a separate package.

Two things in it were added because a run showed they were needed:
- **the refusal branch** in `agent`. On the first live run, the model asked for a tool called `recomment`; Groq refused the call with a 400, and nothing in LangGraph turns that into something the model can correct — the run ended. Chapter 1's `tool_call_failure()` is borrowed to hand Groq's reason back, and the loop goes round again.
- **`INVESTIGATOR_STEPS = 17`**, passed as `recursion_limit` by the desk. LangGraph's own default is 10,007 steps.

```python
# ap_graph.py
def agent(state):
    # Groq validates a tool call before returning it, and refuses one naming a tool that does
    # not exist with a 400. The first live run of notebook 1 met exactly that:
    #     attempted to call tool 'recomment' which was not in request.tools
    # LangChain raises it, `RetryPolicy` rightly will not retry a 400, and nothing in LangGraph's
    # agent pieces turns it into something the model can correct -- so it ended the invoice's
    # run. Chapter 1's `tool_call_failure()` does exactly that, and is borrowed here: the
    # refusal goes back into the conversation and the loop goes round again.
    import openai
    try:
        return {"messages": [bound.invoke(state["messages"])]}
    except openai.BadRequestError as refused:
        from llm import tool_call_failure
        told = tool_call_failure(refused)
        if told is None or state.get("rejected", 0) >= MAX_REJECTIONS:
            raise
        return {"messages": [HumanMessage(told)], "rejected": state.get("rejected", 0) + 1}

def route(state):
    last = state["messages"][-1]
    if isinstance(last, HumanMessage):
        return "agent"                # the provider refused its call; it has been told why
    calls = getattr(last, "tool_calls", None) or []
    if any(c["name"] == "recommend" for c in calls):
        return "finish"
    if calls:
        return "tools"
    return "finish" if state.get("nudged") else "nudge"

def finish(state):
    last = state["messages"][-1]
    call = next((c for c in getattr(last, "tool_calls", []) if c["name"] == "recommend"), None)
    if call is None:
        return {"recommendation": {"action": "hold", "amount": 0.0,
                                   "reason": "The investigator did not hand in a recommendation."}}
    return {"recommendation": P.Recommendation(**call["args"]).model_dump()}
```

The desk calls it from `investigate`, and treats its answer as a proposal: an amount to pay is recomputed by `payables.partial_amount()`, never taken from the model.

```python
# ap_graph.py
def investigate(state):
    invoice = state["invoice"]
    found = "\n".join(f"- {f['kind']}: {f['detail']}" for f in state["findings"])
    try:
        result = investigator.invoke({"messages": [
            SystemMessage(P.INVESTIGATE_PROMPT.format(policy=P.POLICY)),
            HumanMessage(f"Invoice {invoice['invoice_number']} from {invoice['vendor_name']} against "
                         f"{invoice['po_number']} failed the match:\n{found}\n\nInvoice fields: {invoice}")]},
            {"recursion_limit": INVESTIGATOR_STEPS})
        rec = result["recommendation"]
    except GraphRecursionError:
        rec = {"action": "hold", "amount": 0.0,
               "reason": f"The investigator did not finish within {INVESTIGATOR_STEPS} steps."}
    update = {"recommendation": rec, "trail": [f"investigator: {rec['action']} -- {rec['reason']}"]}
    if rec["action"] == "approve_partial":
        # the model proposes, the policy disposes: the amount is recomputed, never trusted
        allowed, qty = P.partial_amount(invoice)
        if abs(rec["amount"] - allowed) > 0.01:
            update["trail"].append(f"investigator proposed {rec['amount']:,.2f}; policy allows {allowed:,.2f}")
        update |= {"amount": allowed, "qty": qty, "signers": P.approvers(invoice, allowed)}
    return update
```

## The wiring

```python
# ap_graph.py, end of build_desk()
graph = StateGraph(Case)
graph.add_node("read_invoice", read_invoice, retry_policy=RETRY)
graph.add_node("check_policy", check_policy)
graph.add_node("answer_duplicate", answer_duplicate)
graph.add_node("verify_bank", verify_bank, destinations=("hold", END))
graph.add_node("ask_clerk", ask_clerk, destinations=("request_approval", "hold"))
graph.add_node("investigate", investigate)
graph.add_node("query_vendor", query_vendor, retry_policy=RETRY)
graph.add_node("await_vendor", await_vendor)
graph.add_node("request_approval", request_approval)
graph.add_node("pay", pay)
graph.add_node("hold", hold)
graph.add_node("refused", refused)

graph.add_edge(START, "read_invoice")
graph.add_edge("read_invoice", "check_policy")
graph.add_conditional_edges("check_policy", after_check,
                            ["request_approval", "pay", "answer_duplicate", "verify_bank",
                             "ask_clerk", "hold", "investigate"])
graph.add_conditional_edges("investigate", after_investigation,
                            ["query_vendor", "request_approval", "pay", "hold"])
graph.add_edge("query_vendor", "await_vendor")
graph.add_edge("await_vendor", "read_invoice")          # the vendor loop: read the reply, check again
graph.add_conditional_edges("request_approval", after_approval, ["pay", "request_approval", "refused"])
for node in ("answer_duplicate", "pay", "hold", "refused"):
    graph.add_edge(node, END)
return graph.compile(checkpointer=checkpointer)
```

`compile(checkpointer=...)` is where the run becomes durable: after every step, the state is written to the checkpointer. A run is identified by its **thread** — `invoice-B` — and a thread is a run you can come back to by name, from anywhere that can reach the same checkpointer.

```python
# ap_graph.py
def start(graph, key, run=""):
    """Begin one invoice. Returns what it is now waiting for, or [] if it finished."""
    graph.invoke({"key": key, "email": P.INBOX[key], "vendor_rounds": 0}, config(key, run))
    return waiting_on(graph, config(key, run))

def answer(graph, key, reply, run=""):
    """Resume a paused invoice with a person's (or a vendor's) reply."""
    graph.invoke(Command(resume=reply), config(key, run))
    return waiting_on(graph, config(key, run))

def settle(graph, key, run="", respond=P.human, max_answers=6):
    """Run one invoice to the end, answering every question from `payables.HUMANS`."""
    asks = start(graph, key, run)
    answered = 0
    while asks and answered < max_answers:
        ask = asks[0]
        asks = answer(graph, key, respond(key, ask["ask"], ask.get("approver")), run)
        answered += 1
    return graph.get_state(config(key, run)).values
```

## The same desk, from scratch: a case file, not a graph

The first version of this build was `ap_graph.py` node for node, with a small graph engine underneath. It worked, and it copied LangGraph's design along with LangGraph's hazards. This repo already has a different answer to "several parties working on one piece of work": chapter 6's **shared board**, where each field has **one owner** who may write it. An invoice case is exactly that, and in accounts payable *who wrote what* is the point:

```python
# ap_scratch.py
OWNER = {"key": "desk", "email": "desk", "next": "desk", "waiting": "desk",
         "invoice": "reader",                     # a model
         "recommendation": "investigator",        # a model with tools
         "findings": "desk", "amount": "desk", "qty": "desk", "signers": "desk", "owner": "desk",
         "approvals": "desk", "outbox": "desk", "vendor_rounds": "desk", "status": "desk", "payment": "desk",
         "callback": "dev", "requester": "dev",   # the AP clerk's answers
         "vendor_reply": "vendor",
         **{f"approval:{person}": person for person in P.PEOPLE}}
```

The models write only what they produce: the reader owns `invoice`, the investigator owns `recommendation`. People write only their own answers: `approval:tomasz` is Tomasz's. **The money is the desk's code alone**: `amount`, `signers`, `status`. A write by anyone else raises `Denied` instead of being recorded, and a field nobody owns cannot be written at all.

`case.py` adds two things to the board. Every write is also a row in chapter 4's `Records`, a SQLite table, and the board is rebuilt from those rows, so any process can carry a case on, and the rows are the audit trail. And steps that wait for people:

```python
# case.py
class Wait:
    """What a step returns when it needs somebody: the field for the answer, who owns it, what comes next."""

    field: str
    party: str
    then: str
    shown: dict = field(default_factory=dict)      # what the person is shown: the question, the packet

    def ask(self):
        return {"field": self.field, "party": self.party, "then": self.then, **self.shown}
```

```python
# case.py
class CaseFile(Shared):
    """A chapter 6 board whose every write is also a row in a chapter 4 `Records` table.

    `Records` has columns for a feed of events, and a write to a board is one: `kind` holds the
    field, `status` who wrote it, `service` which case, `text` the value as JSON.
    """

    def __init__(self, log, thread, **board):
        super().__init__(**board)
        self.log, self.thread = log, thread
        # Rebuilt from the rows, in the order they were written. The owner check is not run again:
        # it ran when each row was written, which is the only time it means anything.
        for row in log.db.execute("SELECT kind, status, text FROM records WHERE service = ? ORDER BY id",
                                  (thread,)):
            before = self.values.get(row["kind"])
            self.values[row["kind"]] = json.loads(row["text"])
            self.history.append({"author": row["status"], "field": row["kind"], "from": before,
                                 "to": self.values[row["kind"]]})

    def set(self, author, field, value):
        if field not in self.owner:
            # chapter 6's board lets anyone write a field nobody owns. A case file holding money does not.
            raise Denied(f"{field} has no owner, so nobody may write it")
        before = super().set(author, field, value)          # raises Denied for anyone but the owner
        self.log.write(datetime.now().isoformat(), field, author, self.thread, json.dumps(value))
        return before
```

**Asking ends a step.** A step that needs a person returns a `Wait`: the field that will hold the answer, the person who owns it, and the step that continues. Answering is that person writing that field. Nothing is re-run to get back to where it was, so nothing is sent twice. Each answer lands in a field named for the person who gives it, so a reordering deploy cannot misfile it, and only that person can write it.

```python
# case.py
def answer(self, thread, party, field, value):
    """`party` answers the case's question by writing `field`. Then the step the question named runs."""
    case = self.case(thread)
    waiting = case.values.get("waiting")
    if not waiting or waiting["field"] != field:
        raise ValueError(f"{thread} is not waiting on {field!r} (waiting on: {waiting})")
    case.set(party, field, value)                       # only the field's owner gets past this
    case.set("desk", "waiting", None)
    case.set("desk", "next", waiting["then"])
    return self._go(case)

def _go(self, case):
    while case.values.get("next") not in (None, END):
        after = self.steps[case.values["next"]](case)
        if isinstance(after, Wait):
            case.set("desk", "waiting", after.ask())
            case.set("desk", "next", None)
            return case.values["waiting"]
        case.set("desk", "next", after or END)
    return None
```

The desk's steps write to the case and say what comes next. Sending to the vendor and waiting for the reply is one step, which is safe here and would be a bug in the graph:

```python
# ap_scratch.py
def query_vendor(case):
    invoice, rec = case.values["invoice"], case.values["recommendation"]
    send(case, invoice["sender_email"], draft(vendor=invoice["vendor_name"], invoice_number=invoice["invoice_number"],
                                              po=invoice["po_number"], reason=rec["reason"]))
    case.set("desk", "vendor_rounds", case.values.get("vendor_rounds", 0) + 1)
    # the email is sent and this step is over; the reply starts `read_reply`, so nothing is sent twice
    return Wait("vendor_reply", "vendor", then="read_reply", shown={
        "ask": "vendor", "to": invoice["sender_email"], "question": "Waiting for a corrected invoice."})

def read_reply(case):
    case.set("desk", "email", case.values["vendor_reply"])
    return "read_invoice"

def request_approval(case):
    signer = _next_signer(case)
    return Wait(f"approval:{signer}", signer, then="record_approval",
                shown={"ask": "approval", "approver": signer, "to": signer, "packet": _packet(case.values)})

def record_approval(case):
    signer = _next_signer(case)
    reply = case.values[f"approval:{signer}"]
    signed = case.values.get("approvals", []) + [{"by": signer, "approved": bool(reply.get("approved")),
                                                  "note": reply.get("note", "")}]
    case.set("desk", "approvals", signed)
    if not signed[-1]["approved"]:
        return close(case, "refused")
    return "pay" if {a["by"] for a in signed} >= set(case.values["signers"]) else "request_approval"
```

The investigator is chapter 6's form-filling agent through `failures.attempt`. If it raises, is stopped by chapter 2's doom-loop check, or answers in prose, it comes back as a **failure form**, and the case is held for a person. Its proposal is stamped with the invoice it is about, so a corrected invoice's approver is never shown the old one's reasons. The amount is recomputed by policy and written by the desk:

```python
# ap_scratch.py
def live_investigate(invoice, findings):
    """Chapter 6's form-filling agent, through `failures.attempt`, so a failure comes back as a form."""
    from failures import attempt
    from registry import Registry
    from supervisor import Agent
    from tools import Tool

    agent = Agent("investigator", P.INVESTIGATE_PROMPT.format(policy=P.POLICY),
                  tools=Registry([Tool(f) for f in P.INVESTIGATOR_TOOLS]), model=MODEL)
    found = "\n".join(f"- {f['kind']}: {f['detail']}" for f in findings)
    request = (f"Invoice {invoice['invoice_number']} from {invoice['vendor_name']} against "
               f"{invoice['po_number']} failed the match:\n{found}\n\nInvoice fields: {invoice}")
    result = attempt(agent, request, model=P.Recommendation)
    if not isinstance(result, P.Recommendation):
        # it raised, Runtime stopped it, or it answered in prose: not a recommendation, so a person looks
        return {"action": "hold", "amount": 0.0, "reason": f"{result.answer}. {result.unknowns}"[:300]}
    return result.model_dump()

def investigate_step(case):
    invoice = case.values["invoice"]
    rec = investigate(invoice, case.values["findings"])
    # stamped with the invoice it is about, so a corrected invoice is never shown the old one's reasons
    case.set("investigator", "recommendation", {**rec, "invoice_number": invoice["invoice_number"]})
    if rec["action"] == "approve_partial":
        # the model proposes, the policy disposes -- and only the desk can write `amount`
        allowed, qty = P.partial_amount(invoice)
        case.set("desk", "amount", allowed)
        case.set("desk", "qty", qty)
        case.set("desk", "signers", P.approvers(invoice, allowed))
        return "request_approval" if case.values["signers"] else "pay"
    if rec["action"] == "query_vendor" and case.values.get("vendor_rounds", 0) < P.MAX_VENDOR_ROUNDS:
        return "query_vendor"
    return "hold"
```

---
# Part 2 · Run and show

In [1]:
import json
import sqlite3
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

from langchain_core.callbacks import get_usage_metadata_callback
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver

import ap_checks
import ap_graph as G
import ap_scratch as A
import ap_stubs as S
import payables as P
from meter import Meter                      # chapter 6's token meter, on the path via ap_scratch

STORE = Path("store")
STORE.mkdir(exist_ok=True)
for old in STORE.glob("ap_*.sqlite*"):
    old.unlink()                              # every run starts from empty records


def scorecard(outcomes):
    good = 0
    for key, got in outcomes.items():
        want = P.EXPECTED[key]
        ok = all(got[f] == want[f] for f in ("status", "amount", "signed_by"))
        good += ok
        print(f"{key}  {got['status']:10} {got['amount']:>10,.2f}  signed by {', '.join(got['signed_by']) or '-':8}"
              f"  {'as policy says' if ok else 'EXPECTED ' + str(want)}")
    print(f"\n{good} of {len(outcomes)} as the policy says")
    return good


def show(result):
    print(json.dumps(result, indent=1, ensure_ascii=False, default=str))

## 1 · Offline, both builds

Every model call replaced by `ap_stubs`, the whole inbox run to the end, every question answered from `payables.HUMANS` — the replies Tomasz, Mira, the clerk and the vendor give. Nothing is spent; this is what makes the live runs below worth paying for.

In [2]:
P.reset()
offline = G.build_desk(InMemorySaver(), extract=S.extract, model=S.Scripted(), draft=S.draft)
print("LangGraph")
scorecard({key: P.outcome(G.settle(offline, key)) for key in P.INBOX})

P.reset()
offline = A.build_desk(extract=S.extract, investigate=S.investigate, draft=S.draft)
print("\nfrom scratch")
scorecard({key: P.outcome(A.settle(offline, key)) for key in P.INBOX})

LangGraph
A  scheduled    3,440.00  signed by -         as policy says
B  scheduled   10,300.00  signed by tomasz    as policy says
C  scheduled   10,320.00  signed by mira      as policy says
D  scheduled    6,525.00  signed by tomasz    as policy says
E  duplicate        0.00  signed by -         as policy says
F  fraud            0.00  signed by -         as policy says
G  scheduled      684.00  signed by mira      as policy says

7 of 7 as the policy says

from scratch
A  scheduled    3,440.00  signed by -         as policy says
B  scheduled   10,300.00  signed by tomasz    as policy says
C  scheduled   10,320.00  signed by mira      as policy says
D  scheduled    6,525.00  signed by tomasz    as policy says
E  duplicate        0.00  signed by -         as policy says
F  fraud            0.00  signed by -         as policy says
G  scheduled      684.00  signed by mira      as policy says

7 of 7 as the policy says


7

## 2 · LangGraph, live

All seven invoices started, nobody answering yet. Each runs until it finishes or has to ask somebody something — and then it is just rows in `store/ap_graph.sqlite`.

In [3]:
desk = G.build_desk(SqliteSaver(sqlite3.connect(STORE / "ap_graph.sqlite", check_same_thread=False)))
P.reset()

cost, waiting = {}, {}
with get_usage_metadata_callback() as usage:
    for key in P.INBOX:
        try:
            waiting[key] = G.start(desk, key)
        except Exception as failed:                  # one invoice failing must not stop the others
            waiting[key] = []
            print(f"{key} FAILED, parked at {desk.get_state(G.config(key)).next}: {type(failed).__name__}: {failed}")
cost["first pass"] = usage.usage_metadata

for key, asks in waiting.items():
    state = desk.get_state(G.config(key)).values
    where = f"waiting on {asks[0]['to']} ({asks[0]['ask']})" if asks else f"finished: {state.get('status')}"
    print(f"{key}  {state['invoice']['invoice_number']:14} {where}")

A  INV-88240      finished: scheduled
B  INV-2026-0917  waiting on tomasz (approval)
C  INV-88251      waiting on ar@brandt-optics.example (vendor)
D  HM-5521        waiting on tomasz (approval)
E  INV-88213      finished: duplicate
F  INV-2026-0931  waiting on dev (callback)
G  PS-3310        waiting on dev (clerk)


What each party is being asked. The approval packets are built by code from the state; the investigator's reasoning and the email to the vendor are the model's.

In [4]:
for key, asks in waiting.items():
    for ask in asks:
        print(f"--- {key} -> {ask['to']} ({ask['ask']})")
        print(ask.get("packet") or ask.get("question"))
        if ask["ask"] == "vendor":
            print("\n  sent to the vendor:\n  " + desk.get_state(G.config(key)).values["outbox"][-1]["body"].replace("\n", "\n  "))
        print()

--- B -> tomasz (approval)
Corvid Electronics Ltd INV-2026-0917 against PO-4480: billed 10,300.00, to pay 10,300.00.

--- C -> ar@brandt-optics.example (vendor)
Waiting for a corrected invoice.

  sent to the vendor:
  Dear Brandt Optics Team,
  
  We have reviewed invoice INV‑88251 that references PO‑4493. According to the purchase order, the LA‑50 units are priced at $86.00 each, and our receiving report confirms that 120 units were delivered. The invoice, however, lists a unit price of $90.30, which exceeds the PO price by more than the permitted 1 % margin.
  
  A corrected invoice should reflect the PO price of $86.00 per unit for the 120 units, resulting in a total amount of $10,320.00.
  
  Could you please issue a revised invoice with these figures and send it to us at your earliest convenience? We will be able to process payment once we receive the corrected document.
  
  Thank you for your prompt attention to this matter.
  
  Kind regards,
  
  Kestrel Labs Accounts Payable

Everyone answers — except Tomasz on invoice **B**, who answers below from a different process.

In [5]:
with get_usage_metadata_callback() as usage:
    for key in "ACDFG":
        asks = waiting[key]
        while asks:
            ask = asks[0]
            asks = G.answer(desk, key, P.human(key, ask["ask"], ask.get("approver")))
cost["after the answers"] = usage.usage_metadata

for key in "CDF":
    print(f"--- {key}")
    for line in desk.get_state(G.config(key)).values["trail"]:
        print("   ", line)

--- C
    read INV-88251 from BRANDT OPTICS GMBH
    check: price
    investigator: query_vendor -- PO-4493 authorizes LA-50 at $86.00 each, and the receipt shows the full 120 units were received. The invoice charges $90.30 per unit, which exceeds the PO price by more than the allowed 1% margin, so the invoice must be corrected before any payment.
    wrote to ar@brandt-optics.example
    vendor replied
    read INV-88251-R from Brandt Optics GMBH
    check: matched
    mira approved
    payment PAY-30201 scheduled: 10,320.00
--- D
    read HM-5521 from Halvorsen Metals
    check: quantity
    investigator: approve_partial -- PO-4485 orders 60 H-220 units at $145 each. Goods receipt shows only 45 units received (backordered 15). No prior billing exists. Invoice HM-5521 bills 60 units, exceeding received quantity, so we approve payment for the 45 units received ($6,525) and defer the remaining $2,175 until delivery.
    tomasz approved
    payment PAY-30202 scheduled: 6,525.00
--- F
   

### Finished by another process

A separate Python process that has never seen this notebook builds the desk, finds `invoice-B` in the SQLite file, and gives Tomasz's answer. The only thing the two processes share is the file.

In [6]:
import os

print(ap_checks.finish_elsewhere(STORE / "ap_graph.sqlite", "B"))
print(f"pid {os.getpid()} (this notebook) reads invoice-B as:", P.outcome(desk.get_state(G.config("B")).values))
print(f"pid {os.getpid()} payments in its ledger:", [(p["invoice"], p["amount"]) for p in P.PAYMENTS])

pid 28896 found invoice-B waiting on: tomasz (approval)
pid 28896 after answering: {'status': 'scheduled', 'amount': 10300.0, 'signed_by': ['tomasz']}
pid 28896 payments in its ledger: [('INV-2026-0917', 10300.0)]

pid 23108 (this notebook) reads invoice-B as: {'status': 'scheduled', 'amount': 10300.0, 'signed_by': ['tomasz']}
pid 23108 payments in its ledger: [('INV-88240', 3440.0), ('INV-88251-R', 10320.0), ('HM-5521', 6525.0), ('PS-3310', 684.0)]


The run crossed processes; **the payment did not.** `P.PAYMENTS` is a list in memory, and invoice B's row exists only in the child's copy, which has exited. The checkpointer makes the *workflow* durable — not the world it changes. A real `pay` writes to a database, and has to be safe to run twice.

In [7]:
graph_outcomes = {key: P.outcome(desk.get_state(G.config(key)).values) for key in P.INBOX}
scorecard(graph_outcomes)
graph_tokens = sum(v["total_tokens"] for usage in cost.values() for v in usage.values())
print(f"\ntokens: {graph_tokens:,}")

A  scheduled    3,440.00  signed by -         as policy says
B  scheduled   10,300.00  signed by tomasz    as policy says
C  scheduled   10,320.00  signed by mira      as policy says
D  scheduled    6,525.00  signed by tomasz    as policy says
E  duplicate        0.00  signed by -         as policy says
F  fraud            0.00  signed by -         as policy says
G  scheduled      684.00  signed by mira      as policy says

7 of 7 as the policy says

tokens: 19,778


## 3 · From scratch, live

The same inbox, the same people, the same model.

In [2]:
P.reset()
scratch = A.build_desk(path=str(STORE / "ap_scratch.sqlite"))
with Meter() as meter:
    scratch_outcomes = {key: P.outcome(A.settle(scratch, key)) for key in P.INBOX}
scratch_cost = meter.summary()

scorecard(scratch_outcomes)
print("\n", scratch_cost)
for key in "CD":
    print(f"--- {key}")
    for line in scratch.state(f"invoice-{key}")["trail"]:
        print("   ", line)

A  scheduled    3,440.00  signed by -         as policy says
B  scheduled   10,300.00  signed by tomasz    as policy says
C  scheduled   10,320.00  signed by mira      as policy says
D  scheduled    6,525.00  signed by tomasz    as policy says
E  duplicate        0.00  signed by -         as policy says
F  fraud            0.00  signed by -         as policy says
G  scheduled      684.00  signed by mira      as policy says

7 of 7 as the policy says

 {'calls': 17, 'prompt': 14951, 'completion': 7923, 'total': 22874, 'seconds': 116}
--- C
    desk: key = C
    desk: email = From: Brandt Optics Accounts <ar@brandt-optics.example> Subject: Invoice INV-88251 BRANDT OPTICS GMBH -- INVOICE INV-...
    desk: vendor_rounds = 0
    reader: invoice = INV-88251 from Brandt Optics GMBH
    desk: findings = price
    investigator: recommendation = query_vendor -- PO-4493 authorizes 120 LA-50 at $86.00 each and the receipt confirms all 120 were received. The invoice charges $90.30 per unit, exceedi

## 4 · What LangGraph does that you have to know

Five behaviours, each reproduced against stand-ins by `ap_checks.py`. The first two fail **silently**.

**A resumed node runs again from its first line.** A node that sends an email and then waits for the reply sends it twice. The same step in the case file sends once:

In [3]:
show(ap_checks.double_send())

{
 "LangGraph": {
  "emails sent while waiting": 1,
  "emails sent after the reply": 2
 },
 "case file": {
  "emails sent while waiting": 1,
  "emails sent after the reply": 1
 }
}


**Answers are matched to questions by position.** Tomasz approves; while the run waits for Ines, a deploy swaps the order of the two `interrupt()` calls; Ines refuses. The case file stores each answer in a field named for its approver:

In [4]:
show(ap_checks.reordered_answers())

{
 "what happened": {
  "tomasz": "approved",
  "ines": "REFUSED"
 },
 "LangGraph recorded": {
  "tomasz": "REFUSED",
  "ines": "approved"
 },
 "case file recorded": {
  "tomasz": "approved",
  "ines": "REFUSED"
 }
}


**Anyone can answer.** `Command(resume=...)` resumes the run with whatever it is given; who is answering is the application's to check, and `ap_graph.py`, like most code written against `interrupt()`, does not. Invoice B waits for Tomasz, and Mira answers it:

In [5]:
show(ap_checks.wrong_person())

{
 "waiting on": "approval:tomasz",
 "LangGraph, after Mira answered": {
  "status": "scheduled",
  "amount": 10300.0,
  "signed_by": [
   "tomasz"
  ]
 },
 "case file, when Mira answered": "Denied: approval:tomasz is tomasz's to set, not mira's",
 "case file, still waiting on": "approval:tomasz"
}


**The default step limit is 10,007.** An investigator that calls the same tool forever, in both builds:

In [6]:
show(ap_checks.runaway_agent())

{
 "LangGraph default recursion_limit": 10007,
 "LangGraph, limit 17": {
  "status": "held",
  "amount": 0.0,
  "signed_by": [],
  "model calls": 9,
  "why": "investigator: hold -- The investigator did not finish within 17 steps."
 },
 "scratch": {
  "status": "held",
  "amount": 0.0,
  "signed_by": [],
  "model calls": 3,
  "why": "Could not do this: investigator was stopped before finishing. the same tool call with identical arguments repeated. This is its best answer from what it had gat"
 }
}


Measured while building this: with the default limit, the same agent ran **5,004 model calls** (92 seconds against a stand-in) before LangGraph stopped it. Chapter 2's `Runtime` stops on the *second identical call* — doom-loop detection — and gives the agent one turn without tools to say what it found. A count cannot tell an agent working hard from one going round in circles.

**A loop through `interrupt()` is never stopped by `recursion_limit`**, because the limit is per `invoke` and every resume is a new `invoke`. With the round counter taken out of the state, a vendor re-sending the same wrong invoice is written to forever:

In [12]:
show(ap_checks.endless_vendor_loop())

{
 "vendor rounds": 21,
 "emails sent": 21,
 "still waiting for the vendor": true
}


**Retries are layered.** `RetryPolicy(max_attempts=3)` on a node, over the OpenAI SDK's `max_retries=8` — `3 × (1 + 8) = 27` requests per node before anything gives up. One 503, recovered:

In [13]:
show(ap_checks.flaky_provider())

{
 "status": "scheduled",
 "amount": 3440.0,
 "signed_by": [],
 "attempts at reading": 2
}


**A tool call the provider refuses.** The failure from the first live run, reproduced with Groq's exact response body. Without the fix the run ends — and LangGraph leaves the thread parked at the last good checkpoint, so `invoke(None, config)` carries on from there. With the fix, the refusal goes back to the model:

In [14]:
show(ap_checks.refused_tool_call())

{
 "without the fix, the run ended with": "OpenAIInvalidRequestError",
 "thread parked before": [
  "investigate"
 ],
 "invoke(None) then reached": [
  "vendor"
 ],
 "with the fix, the investigator's conversation": [
  [
   "HumanMessage",
   "Invoice INV-88251 from Brandt Optics against PO-4493 failed the match: price 90.30 against 86.00."
  ],
  [
   "HumanMessage",
   "Your last tool call was rejected: Tool call validation failed: tool call validation failed: attempted to call tool 'recomment' which was not"
  ],
  [
   "AIMessage",
   "lookup_po({'po_number': 'PO-4493'})"
  ],
  [
   "ToolMessage",
   "PO-4493: Brandt Optics GmbH, raised 2026-08-27, owner Mira Okafor (optics lead, budget owner). LA-50 lens assembly, 50 mm: 120 ordered at 86"
  ],
  [
   "AIMessage",
   "recommend({'action': 'query_vendor', 'amount': 0.0, 'reason': 'Billed 90.30 per LA-50; PO-4493 agreed 86.00.'})"
  ]
 ],
 "recommendation": "query_vendor"
}


## 5 · Correcting a run after the fact

Invoice A read with the last two digits of its IBAN swapped. The policy freezes it and asks for a callback — correctly. The clerk sees the email says `31`, rewinds to the step after reading, fixes the field, and lets the policy run again. `update_state` writes a **new** checkpoint branching off the old one; the frozen run is still in the history.

In [15]:
show(ap_checks.misread_iban())

{
 "waiting on": [
  "callback"
 ],
 "because": "asks for payment to 'IBAN DE44 5001 0517 5407 3249 13'; on file: DE44 5001 0517 5407 3249 31",
 "steps before the fix": [
  [
   -1,
   [
    "__start__"
   ]
  ],
  [
   0,
   [
    "read_invoice"
   ]
  ],
  [
   1,
   [
    "check_policy"
   ]
  ],
  [
   2,
   [
    "verify_bank"
   ]
  ]
 ],
 "after the fix": {
  "status": "scheduled",
  "amount": 3440.0,
  "signed_by": []
 },
 "trail": [
  "read INV-88240 from Brandt Optics GmbH",
  "clerk corrected a misread IBAN",
  "check: matched",
  "payment PAY-30200 scheduled: 3,440.00"
 ],
 "checkpoints in the thread": 7
}


## 6 · Side by side

In [3]:
import ast
import io
import tokenize


def code_lines(path):
    # lines holding code: not blank, not a comment, not a docstring
    source = Path(path).read_text(encoding="utf-8")
    docstrings = set()
    for node in ast.walk(ast.parse(source)):
        body = getattr(node, "body", None)
        if isinstance(body, list) and body and isinstance(body[0], ast.Expr) \
                and isinstance(getattr(body[0], "value", None), ast.Constant) and isinstance(body[0].value.value, str):
            docstrings.update(range(body[0].lineno, body[0].end_lineno + 1))
    lines = set()
    for token in tokenize.generate_tokens(io.StringIO(source).readline):
        if token.type not in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT,
                              tokenize.DEDENT, tokenize.ENDMARKER):
            lines.update(l for l in range(token.start[0], token.end[0] + 1) if l not in docstrings)
    return len(lines)


good = lambda outcomes: sum(all(outcomes[k][f] == P.EXPECTED[k][f] for f in ("status", "amount", "signed_by"))
                            for k in outcomes)
print(f"{'':14}{'as policy says':>16}{'tokens':>10}{'lines of code':>15}")
print(f"{'LangGraph':14}{good(graph_outcomes):>14}/7{graph_tokens:>10,}{code_lines('ap_graph.py'):>15}")
print(f"{'from scratch':14}{good(scratch_outcomes):>14}/7{scratch_cost['total']:>10,}"
      f"{code_lines('ap_scratch.py') + code_lines('case.py'):>15}  (ap_scratch {code_lines('ap_scratch.py')} + case {code_lines('case.py')})")

                as policy says    tokens  lines of code
LangGraph                  7/7    19,778            267
from scratch               7/7    22,874            263  (ap_scratch 172 + case 91)


## Findings

**Both builds got all seven invoices right, at about the same cost and size.** 19,778 tokens against 22,874, one live run each, and 17 model calls in the scratch run, as in its first design: the same prompts to the same model, so the framework does not change what the model is asked. 267 lines of code against 263. The difference is in what each design makes easy, and what it makes impossible.

**Two designs, not one design twice.** A graph is a picture of the *process*: nodes, edges, a checkpoint of the whole state after each step. The case file is a picture of the *case*: fields, each with one owner, and a log of who wrote what. Both pause for days and resume in another process. They fail differently:

| | LangGraph | Case file (this repo's parts) |
|---|---|---|
| the state | a `TypedDict`, reducers to merge updates | chapter 6's board: a field per fact, **one owner per field** |
| who may change the amount | any node | the desk's code; a model or a person writing it is `Denied` |
| persistence | a checkpoint of the whole state per step | each write a row in chapter 4's `Records` — the audit trail, with authors |
| waiting for a person | `interrupt()`; resuming **re-runs the node** from its first line | a `Wait` **ends** the step; the answer starts the next one |
| an email sent before a wait, in the same step | sent twice | sent once |
| several approvals, reordered by a deploy | matched by **position**: the answers swap | each answer is its approver's field: nothing to swap |
| an approval answered by the wrong person | recorded as the right person's | refused; the case still waits |
| a runaway agent | a step count, default **10,007**; held after 9 calls at our limit | chapter 2's doom-loop check; held after **3** |
| an agent that fails, stops, or answers in prose | handled node by node | chapter 6's `failures.attempt`: a failure form, then hold |
| rewinding and correcting a past step | `get_state_history`, `update_state` forks | not built |
| parallel steps, streaming, caching, timeouts, a server | yes | not built |

**Where LangGraph earns its place**: the bottom two rows, and durability you do not have to write. Pausing, resuming in another process, carrying on after a crash, rewinding and correcting all worked the first time. Rewind-and-fork is the one capability here that this repo's parts do not give you cheaply: a log of writes can say *what happened*, but replaying a case from the middle with a field changed is a graph's natural move, not a board's.

**Where the case file earns its place**: the hazards above are not bugs in either build. They follow from each design. A graph that resumes by re-running a node will repeat whatever the node did before it paused, and will match answers by order, unless every developer remembers not to let it. A case file where asking ends a step, and every field has an owner, cannot do either. And it says *who* did each thing, which is what an accounts-payable audit asks first.

**What you have to know about LangGraph before production**, all reproduced above for free: a resumed node runs again from its first line, so a side effect never shares a node with an `interrupt()` after it; answers are matched by position, so never reorder `interrupt()` calls under paused runs; `Command(resume=...)` takes whatever it is given, so check who is answering; the default step limit is ten thousand, so every agent loop needs its own; a loop through `interrupt()` is bounded only by a counter you keep; retry layers multiply. And one found live: the provider refusing a tool call ends the run unless you catch it.

**Durability stops at the workflow, in both.** Invoice B was finished by another process and reads as scheduled here; its payment exists only in that process's memory. A checkpoint or a case file saves what the run knows, not what it did to the world. A real `pay` writes to a database, and has to be safe to run twice.

**What neither build checks: the words.** Every figure the policy acts on is recomputed in code: the investigator proposed 6,525.00, and `partial_amount()` confirmed it rather than trusting it. The email to the vendor is the model's, sent as written. This run's was right; in an earlier live run it asked Brandt for a corrected invoice at **€86.00** a unit, though the invoice and PO are in dollars. A check on outgoing drafts, reading figures and currency back against the record (chapter 3's `verified()` again), is the missing piece in both builds.